# 04c. Extension Intraday (1h) : Perpétuel `GOLD-USDC` (`xyz:GOLD`) sur Hyperliquid

**Projet :** Prévision de l'or — *Advanced Time Series* (ENSAI, 3A)

### Objectifs de cette extension :
1. Analyser la série haute fréquence (**1h**, 5 000 bougies 24/7 de mai à novembre 2026) du contrat perpétuel `xyz:GOLD` sur **Hyperliquid** (`data/hyperliquid_gold_1h.csv`).
2. Intégrer des **indicateurs techniques stationnarisés** (**RSI 14h**, **MACD normalisé**, **ATR/Range**) et des **variables de microstructure on-chain** (**Funding Rate**, **Premium**, **Nombre de trades**, **Volume**).
3. Comparer **Naïf**, **ARIMA**, **Gradient Boosting**, **GARCH(1,1)** et **Chronos-2 Zero-Shot** en validation *walk-forward* sur les **200 dernières heures** (rendements $r_t$ et volatilité $|r_t|$).

In [ ]:
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from chronos import BaseChronosPipeline

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement, EDA horaire & Indicateurs Techniques / On-Chain

In [ ]:
df = pd.read_csv(DATA_DIR / "hyperliquid_gold_1h.csv", parse_dates=["Date"])
df = df.sort_values("Date").set_index("Date").asfreq("1h").ffill()

# Cibles : log-rendement horaire et amplitude (proxy de volatilité)
df["return"] = np.log(df["Close"]).diff()
df["abs_return"] = np.abs(df["return"])

# Indicateurs techniques stationnarisés (calculés sur le passé jusqu'à t)
delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / loss.replace(0, np.nan)
df["RSI_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD_norm"] = (ema12 - ema26) / ema26
df["MACD_signal_norm"] = df["MACD_norm"].ewm(span=9, adjust=False).mean()
df["MACD_hist_norm"] = df["MACD_norm"] - df["MACD_signal_norm"]

df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
df["vol_12h"] = df["return"].rolling(12).std()
df["vol_24h"] = df["return"].rolling(24).std()
df["log_trades"] = np.log1p(df["Trades"])
df["log_volume"] = np.log1p(df["Volume"])
df["hour_utc"] = df.index.hour
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

df = df.dropna().copy()
print(f"Dataset 1h prêt : {len(df)} heures | {df.index.min()} -> {df.index.max()}")

# Statistiques descriptives 1h
r = df["return"]
adf_p = adfuller(r)[1]
jb_stat, jb_p = stats.jarque_bera(r)
print(f"Skewness: {stats.skew(r):.3f} | Excès Kurtosis: {stats.kurtosis(r):.2f} | JB p-val: {jb_p:.2e} | ADF p-val: {adf_p:.2e}")

## 2. Préparation des features décalées ($t-1$) pour prévoir $t$ (zéro fuite)

In [ ]:
feature_cols_raw = [
    "return", "abs_return", "RSI_14", "MACD_norm", "MACD_hist_norm",
    "range_pct", "vol_12h", "vol_24h", "FundingRate", "Premium",
    "log_trades", "log_volume"
]

X_lags = pd.DataFrame(index=df.index)
for col in feature_cols_raw:
    X_lags[f"{col}_lag1"] = df[col].shift(1)
for k in [2, 3, 6, 12, 24]:
    X_lags[f"return_lag{k}"] = df["return"].shift(k)
    X_lags[f"abs_return_lag{k}"] = df["abs_return"].shift(k)
X_lags["hour_utc"] = df["hour_utc"]
X_lags["is_weekend"] = df["is_weekend"]

data_ml = X_lags.join(df[["return", "abs_return"]]).dropna()
feat_names = [c for c in data_ml.columns if c not in ["return", "abs_return"]]

N_TEST = 200  # ~8.3 jours d'heures continues hors échantillon
train_df = data_ml.iloc[:-N_TEST]
test_df = data_ml.iloc[-N_TEST:]
print(f"Train: {len(train_df)}h | Test: {len(test_df)}h")

## 3. Comparaison des modèles sur les **rendements horaires ($r_t$)**

In [ ]:
y_train = train_df["return"].values
y_test = test_df["return"].values

# 1. Naïf (r_t = 0)
pred_naive = np.zeros_like(y_test)

# 2. ARIMA(1,0,1) estimé sur train puis filtré en walk-forward
arima_fit = ARIMA(y_train, order=(1, 0, 1)).fit()
full_series = pd.concat([train_df["return"], test_df["return"]])
arima_applied = ARIMA(full_series, order=(1, 0, 1)).filter(arima_fit.params)
pred_arima = arima_applied.fittedvalues.iloc[-N_TEST:].values

# 3. Gradient Boosting (Technique + Microstructure Hyperliquid)
gb_ret = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42)
gb_ret.fit(train_df[feat_names], y_train)
pred_gb = gb_ret.predict(test_df[feat_names])

# 4. Chronos-2 Zero-Shot (Univarié sur les rendements 1h)
pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
CTX = 512
df_chronos_src = df[["return", "abs_return", "FundingRate", "Premium", "RSI_14", "range_pct"]].reset_index()
start_pos = len(df_chronos_src) - N_TEST

win_uni, win_cov = [], []
cov_c = ["FundingRate", "Premium", "RSI_14", "range_pct"]
for i in range(N_TEST):
    t = start_pos + i
    s = df_chronos_src.iloc[t - CTX : t][["Date", "return"] + cov_c].copy()
    s["item_id"] = f"w_{i:03d}"
    win_cov.append(s)
    win_uni.append(s[["item_id", "Date", "return"]])

wf_uni = pipeline.predict_df(pd.concat(win_uni), prediction_length=1, quantile_levels=[0.5], id_column="item_id", timestamp_column="Date", target="return", freq="1h", batch_size=32)
wf_cov = pipeline.predict_df(pd.concat(win_cov), prediction_length=1, quantile_levels=[0.5], id_column="item_id", timestamp_column="Date", target="return", freq="1h", batch_size=32)
pred_c_uni = wf_uni.sort_values("item_id")["0.5"].values
pred_c_cov = wf_cov.sort_values("item_id")["0.5"].values

def row_m(name, y_t, y_p):
    return {
        "Modèle": name,
        "MAE": mean_absolute_error(y_t, y_p),
        "RMSE": root_mean_squared_error(y_t, y_p),
        "Hit Ratio": np.mean(np.sign(y_t) == np.sign(y_p)) if np.any(y_p != 0) else 0.50
    }

df_res_ret = pd.DataFrame([
    row_m("Naïf (r_t = 0)", y_test, pred_naive),
    row_m("ARIMA(1,0,1)", y_test, pred_arima),
    row_m("Gradient Boosting (Tech + HL)", y_test, pred_gb),
    row_m("Chronos-2 Univarié", y_test, pred_c_uni),
    row_m("Chronos-2 + HL Covariates", y_test, pred_c_cov),
])
print("=== RÉSULTATS SUR RENDEMENTS HORAIRES (1h) ===")
print(df_res_ret.to_string(index=False))

## 4. Comparaison sur la **volatilité horaire ($|r_t|$)** : GARCH vs ML vs Chronos-2

In [ ]:
v_test = test_df["abs_return"].values
v_naive = test_df["abs_return_lag1"].values

# GARCH(1,1) Student-t sur rendements * 1000 (échelle optimale pour l'optimiseur arch)
scale_factor = 1000.0
am = arch_model(full_series * scale_factor, vol="Garch", p=1, q=1, dist="t")
garch_res = am.fit(last_obs=len(train_df), disp="off")
garch_fc = garch_res.forecast(start=len(train_df), horizon=1, reindex=False)
sigma_garch = np.sqrt(garch_fc.variance.iloc[-N_TEST:, 0].values) / scale_factor
# E[|r_t|] ~= sigma_t * sqrt(2/pi)
pred_garch_abs = sigma_garch * np.sqrt(2 / np.pi)

# Gradient Boosting sur |r_t|
gb_vol = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=150, min_samples_leaf=50, random_state=42)
gb_vol.fit(train_df[feat_names], train_df["abs_return"].values)
pred_gb_vol = np.clip(gb_vol.predict(test_df[feat_names]), 0, None)

# Chronos-2 Zero-Shot sur |r_t|
win_vol = []
for i in range(N_TEST):
    t = start_pos + i
    s = df_chronos_src.iloc[t - CTX : t][["Date", "abs_return"]].copy()
    s["item_id"] = f"w_{i:03d}"
    win_vol.append(s)
wf_vol = pipeline.predict_df(pd.concat(win_vol), prediction_length=1, quantile_levels=[0.5], id_column="item_id", timestamp_column="Date", target="abs_return", freq="1h", batch_size=32)
pred_c_vol = np.clip(wf_vol.sort_values("item_id")["0.5"].values, 0, None)

df_res_vol = pd.DataFrame([
    {"Modèle": "Naïf (|r_{t-1}|)", "MAE": mean_absolute_error(v_test, v_naive), "RMSE": root_mean_squared_error(v_test, v_naive)},
    {"Modèle": "GARCH(1,1)-t", "MAE": mean_absolute_error(v_test, pred_garch_abs), "RMSE": root_mean_squared_error(v_test, pred_garch_abs)},
    {"Modèle": "Gradient Boosting (Tech + HL)", "MAE": mean_absolute_error(v_test, pred_gb_vol), "RMSE": root_mean_squared_error(v_test, pred_gb_vol)},
    {"Modèle": "Chronos-2 Zero-Shot", "MAE": mean_absolute_error(v_test, pred_c_vol), "RMSE": root_mean_squared_error(v_test, pred_c_vol)},
])
print("\n=== RÉSULTATS SUR VOLATILITÉ HORAIRE |r_t| (1h) ===")
print(df_res_vol.to_string(index=False))

plt.figure(figsize=(13, 4))
plt.plot(test_df.index, v_test, label="Réalisé |r_t|", color="black", alpha=0.5)
plt.plot(test_df.index, pred_gb_vol, label="Gradient Boosting (Tech + HL)", color="#d62728", lw=1.5)
plt.plot(test_df.index, pred_garch_abs, label="GARCH(1,1)-t", color="#1f77b4", lw=1.5)
plt.plot(test_df.index, pred_c_vol, label="Chronos-2 Zero-Shot", color="#2ca02c", lw=1.5)
plt.title("Hyperliquid xyz:GOLD (1h) — Prévision de la volatilité |r_t|", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "04c_hyperliquid_1h_vol.png", dpi=300)
plt.show()